In [31]:
# import boto3
import sagemaker

import boto3
from sagemaker.pytorch import PyTorch
from sagemaker.pytorch.processing import PyTorchProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

In [32]:
%pip install -U "botocore[crt]" awscrt

/bin/bash: /home/aman/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  Using cached awscrt-0.32.1-cp313-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (10 kB)
Note: you may need to restart the kernel to use updated packages.


In [33]:
# Configuration
REGION = "us-east-1"
ROLE_ARN = (
    "arn:aws:iam::253490779227:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1"
)
BUCKET = "autonomous-bike"
S3_IMAGES = f"s3://{BUCKET}/data/100k_images/"
S3_ANNOTATIONS = f"s3://{BUCKET}/data/100k/"

In [34]:
boto_session = boto3.Session(region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

In [35]:
!python manage_sagemaker_jobs.py

/bin/bash: /home/aman/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
python: can't open file '/home/aman/Projects/Autonomous-Bicycle/manage_sagemaker_jobs.py': [Errno 2] No such file or directory


In [38]:
train_file = "train.py"

# Storage configuration:
# - Model artifacts (.pth files) -> output_path
# - Metrics (JSON/CSV) -> output_data_config

models = ["autonomous-bike", "autonomous-bike-real"]

model_output_path = f"s3://{BUCKET}/{models[-1]}_model"
metrics_output_path = f"s3://{BUCKET}/{models[-1]}_output"

estimator_3 = PyTorch(
    entry_point=train_file,
    dependencies=[
        "requirements.txt",
    ],
    role=ROLE_ARN,
    framework_version="2.4",
    py_version="py311",
    output_path=model_output_path,
    code_location=model_output_path,
    output_data_config={"S3OutputPath": metrics_output_path},
    instance_count=1,
    instance_type="ml.g6e.2xlarge",
    volume_size=100,
    max_run=86400,
    hyperparameters={
        "train-split": "train",
        "test-split": "val",
        "image-height": 512,
        "image-width": 1024,
        "epochs": 20,
        "batch-size": 8,
        "test-batch-size": 8,
        "num-workers": 8,
        "backbone": "resnet101",
        "pretrained-backbone": True,
        "base-lr": 0.0035,
        "max-iters": 175000,
        "momentum": 0.9,
        "weight-decay": 1e-4,
        "lr-power": 0.9,
        "train-hflip-prob": 0.5,
        "amp": True,
        "sync-bn": False,
        "progress": True,
        "best-metric": "primary_class_iou",
        "primary-class-name": "area/drivable",
        "category-map-path": "configs/drivable_only_category_map.json",
        "no-auto-discover-classes": True,
        "save-file": f"{models[-1]}Weights.pth",
    },
#     hyperparameters={
#       "train-split": "train",
#       "test-split": "val",
#       "image-height": 256,
#       "image-width": 512,
#       "epochs": 1,
#       "batch-size": 2,
#       "test-batch-size": 2,
#       "num-workers": 2,
#       "backbone": "resnet50",
#       "base-lr": 0.001,
#       "max-iters": 20,
#       "momentum": 0.9,
#       "weight-decay": 1e-4,
#       "lr-power": 0.9,
#       "train-hflip-prob": 0.0,
#       "amp": False,
#       "sync-bn": False,
#       "progress": True,
#       "save-file": f"{models[-1]}_smoke_test.pth",
#   },

    
    sagemaker_session=sagemaker_session,
    base_job_name=models[-1],
)

print("Estimator configured:")
# print(f"  Instance -> ml.g6e.2xlarge")
# print(f"  Source bundle -> {model_output_path}")
# print(f"  Model artifacts -> {model_output_path}")
# print(f"  Metrics -> {metrics_output_path}")


Estimator configured:


In [ ]:
estimator_3.fit(
    {"images": S3_IMAGES, "annotations": S3_ANNOTATIONS},
    wait=True,  # Wait for job to complete
    logs="All",  # Stream all logs to the notebook
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: autonomous-bike-real-2026-04-19-03-19-19-513


2026-04-19 03:19:21 Starting - Starting the training job
2026-04-19 03:19:21 Pending - Training job waiting for capacity............
2026-04-19 03:21:10 Pending - Preparing the instances for training...
2026-04-19 03:21:51 Downloading - Downloading input data.........
2026-04-19 03:23:27 Downloading - Downloading the training image............
2026-04-19 03:25:23 Training - Training image download completed. Training in progress..

In [ ]:
train_file = "dss_transformer_train.py"
part1_my_output_path = (
    "s3://sagemaker-us-west-1-253490779227/animal-classification-resnet18"
)
gpu = "ml.g4dn.2xlarge"
estimator = PyTorch(
    entry_point=train_file,
    output_path=part1_my_output_path,
    dependencies=["requirements.txt"],
    role=ROLE_ARN,
    framework_version="2.1",
    py_version="py310",
    instance_count=1,
    instance_type=gpu,
    hyperparameters={
        "epochs": 10,
        "batch-size": 64,
        "learning-rate": 1e-5,
        "use-cuda": True,
        "image-size": 224,
        "weight-decay": 1e-8,
        "stochastic-depth": 0.2,
        "num-cpu": 4,
        "save-file": "resnet18_model.pth",
    },
    sagemaker_session=sagemaker_session,
    base_job_name="resnet18",
    # max_run=3600,
)
# estimator.latest_training_job.stop()
print(estimator)

In [6]:
train_file = "dss_transformer_train.py"
part1_my_output_path = (
    "s3://sagemaker-us-west-1-253490779227/animal-classification-models_part1"
)

estimator = PyTorch(
    entry_point=train_file,
    output_path=part1_my_output_path,
    dependencies=["requirements.txt"],
    role=ROLE_ARN,
    framework_version="2.1",
    py_version="py310",
    instance_count=1,
    instance_type="ml.g4dn.xlarge",  # GPU instance with NVIDIA T4
    hyperparameters={
        "epochs": 10,
        "batch-size": 64,
        "learning-rate": 1e-5,
        "use-cuda": True,
        "image-size": 224,
        "weight-decay": 1e-8,
        "stochastic-depth": 0.2,
        "num-cpu": 4,
        "save-file": "final_swin_t_model_part1.pth",
    },
    sagemaker_session=sagemaker_session,
    base_job_name="swin-stage1",
    # max_run=3600,
)
# estimator.latest_training_job.stop()
print(estimator)

In [7]:
print(
    "We are training using this file: ",
    train_file,
    " with this data: ",
    S3_PREPROCESSED,
)

estimator.fit(
    {"training": S3_PREPROCESSED},
    wait=True,  # ✅ Wait for job to complete
    logs="All",  # ✅ Stream ALL logs to notebook (shows all print statements!)
)

We are training using this file:  dss_transformer_train.py  with this data:  s3://animal-classification-virgina/processed


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │   S3_PREPROCESSED,                                                                        │
│    6 )                                                                                           │
│    7                                                                                             │
│ ❱  8 estimator.fit(                                                                              │
│    9 │   {"training": S3_PREPROCESSED},                                                          │
│   10 │   wait=True,  # ✅ Wait for job to complete                                               │
│   11 │   logs="All",  # ✅ Stream ALL logs to notebook (shows all print statements!)             │
│                                                                                                  │
│ /home/aman/Projects/DSS-Image-Classification/DSSAnimalClassification/.venv/lib/python3.13/site-p │
│ ackages/sagemaker/pytorch/estimator.py:826 in fit                                                │
│                                                                                                  │
│    823 │   │   │   │   │   "sagemaker_recipe_local_path": f"/opt/ml/input/data/{recipe_channel_  │
│    824 │   │   │   │   }                                                                         │
│    825 │   │   │   )                                                                             │
│ ❱  826 │   │   return super(PyTorch, self).fit(                                                  │
│    827 │   │   │   inputs=inputs,                                                                │
│    828 │   │   │   wait=wait,                                                                    │
│    829 │   │   │   logs=logs,                                                                    │
│                                                                                                  │
│ /home/aman/Projects/DSS-Image-Classification/DSSAnimalClassification/.venv/lib/python3.13/site-p │
│ ackages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper                                  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /home/aman/Projects/DSS-Image-Classification/DSSAnimalClassification/.venv/lib/python3.13/site-p │
│ ackages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper                                  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)   

In [ ]:
print(estimator.model_data)

In [ ]:
# ========================================
# RUN THIS CELL TO LIST ALL RECENT JOBS
# ========================================

import boto3
from datetime import datetime

sagemaker_client = boto3.client("sagemaker", region_name=REGION)

# Get recent training jobs
jobs = sagemaker_client.list_training_jobs(
    MaxResults=10, SortBy="CreationTime", SortOrder="Descending"
)

print("=" * 80)
print("RECENT TRAINING JOBS")
print("=" * 80)
print(f"\n{'#':<4} {'Job Name':<50} {'Status':<15}")
print("-" * 80)

for i, job in enumerate(jobs["TrainingJobSummaries"]):
    job_name = job["TrainingJobName"]
    status = job["TrainingJobStatus"]
    created = job["CreationTime"].strftime("%Y-%m-%d %H:%M")

    # Color code status
    status_symbol = {
        "InProgress": "🔄",
        "Completed": "✅",
        "Failed": "❌",
        "Stopped": "⏸️",
    }.get(status, "❓")

    print(f"{i:<4} {job_name:<50} {status_symbol} {status}")

print("\n" + "=" * 80)
print("💡 Copy a job name above and paste it in the next cell to view its logs")
print("=" * 80)

In [ ]:
# ========================================
# RUN THIS CELL TO VIEW LOGS FROM A JOB
# Paste the job name from above, or it will use the most recent job
# ========================================

import boto3


def get_training_logs(job_name, region="us-west-1", max_lines=None):
    """
    Get all CloudWatch logs for a SageMaker training job

    Args:
        job_name: Name of the training job
        region: AWS region
        max_lines: Maximum number of log lines to show (None = all)
    """
    logs_client = boto3.client("logs", region_name=region)

    log_group = "/aws/sagemaker/TrainingJobs"

    try:
        # List all log streams for this job
        streams = logs_client.describe_log_streams(
            logGroupName=log_group,
            logStreamNamePrefix=job_name,
            orderBy="LogStreamName",
        )

        if not streams["logStreams"]:
            print(f"❌ No logs found for job: {job_name}")
            print("   Job might still be starting, or name is incorrect")
            return

        all_logs = []
        for stream in streams["logStreams"]:
            stream_name = stream["logStreamName"]

            # Get all events from this stream
            next_token = None
            while True:
                kwargs = {
                    "logGroupName": log_group,
                    "logStreamName": stream_name,
                    "startFromHead": True,
                }
                if next_token:
                    kwargs["nextToken"] = next_token

                response = logs_client.get_log_events(**kwargs)

                for event in response["events"]:
                    all_logs.append(event["message"])

                # Check if there are more logs
                next_token = response.get("nextForwardToken")
                if not response["events"] or next_token == kwargs.get("nextToken"):
                    break

        # Print logs
        print("=" * 80)
        print(f"📋 TRAINING LOGS: {job_name}")
        print("=" * 80)
        print(f"Total log lines: {len(all_logs)}")
        if max_lines:
            print(f"Showing first {max_lines} lines (set max_lines=None for all)")
        print("=" * 80)
        print()

        logs_to_show = all_logs[:max_lines] if max_lines else all_logs
        for log in logs_to_show:
            print(log)

        if max_lines and len(all_logs) > max_lines:
            print()
            print("=" * 80)
            print(f"⚠️  Showing {max_lines} of {len(all_logs)} total lines")
            print(f"   Run: get_training_logs('{job_name}', max_lines=None) to see all")
            print("=" * 80)

    except Exception as e:
        print(f"❌ Error retrieving logs: {e}")
        print(f"   Make sure the job name is correct")


# ========================================
# PASTE JOB NAME HERE (or leave empty for most recent)
# ========================================
JOB_NAME = ""  # Example: 'animal-classification-training-2024-12-21-12-34-56-789'

# If no job name provided, use most recent
if not JOB_NAME:
    sagemaker_client = boto3.client("sagemaker", region_name=REGION)
    jobs = sagemaker_client.list_training_jobs(
        MaxResults=1, SortBy="CreationTime", SortOrder="Descending"
    )
    if jobs["TrainingJobSummaries"]:
        JOB_NAME = jobs["TrainingJobSummaries"][0]["TrainingJobName"]
        print(f"ℹ️  No job name specified, using most recent: {JOB_NAME}\n")
    else:
        print("❌ No training jobs found")

# View logs (showing first 500 lines by default)
if JOB_NAME:
    get_training_logs(JOB_NAME, REGION, max_lines=500)

# 💡 To see ALL logs without limit:
# get_training_logs(JOB_NAME, REGION, max_lines=None)

---

## 📖 Quick Reference: Which Cells to Run

### **To Start Training (First Time):**
1. ✅ Run **Cells 0-3** (Setup & Config)
2. ✅ Run **Cell 4-5** (Preprocessing) - Only once needed
3. ✅ Run **Cell 6** (Create Estimator)
4. ✅ Run **Cell 7** (Start Training) - **SEE ALL LOGS HERE!**

### **To View Logs After Closing Computer:**
1. ✅ Run **Cells 0-3** (Setup & Config)
2. ✅ Run **Cell 9** (List all recent jobs)
3. ✅ Run **Cell 10** (View logs fromselected job)

### **Tips:**
- **Cell 7** shows ALL your `print()` statements from `dss_train.py` in real-time
- Logs are automatically saved to CloudWatch (accessible anytime!)
- Safe to close computer during training - logs persist in CloudWatch
- **Cell 10** retrieves logs from CloudWatch whenever you need them

---
